Descripción...
<br>
- Agrupa archivos procesados por período y condiciones de toma de datos.
- Taggea eventos por partícula y región del detector.
- Guarda el H5 file agrupado.

In [1]:
import sys
sys.path.append('/lhome/ific/c/ccortesp/Analysis/')

from libs import bckg_functions as bf
from libs import crudo
from libs import fit_functions as ff
from libs import plotting_tools as pt

import argparse
import csv
import glob
from invisible_cities.reco.corrections import read_maps, apply_all_correction
from invisible_cities.types.symbols import NormStrategy
from invisible_cities.core.core_functions import in_range
from joblib import Parallel, delayed 
import numpy as np
import os
import pandas as pd
from scipy.interpolate import interp1d
from scipy.interpolate import griddata
from scipy.spatial.distance import cdist
from sklearn.neighbors import BallTree
from sklearn.neighbors import NearestNeighbors
from sklearn.exceptions import NotFittedError
from typing import List, Callable, Tuple

%matplotlib inline
%load_ext autoreload
%autoreload 2

# Configuration

In [2]:
# --------------------------------------------
# 1. DIRECTORIES, PATHS, KEYS AND FILENAMES
# --------------------------------------------
# FILENAME TAG
VERSION_TAG = 'p2_final'

# DIRECTORIES, PATHS & FILES
DATA_DIR   = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Sophronia/Low_background/'
ICAROS_DIR = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Icaros/Low_background/'
PROCESSED_DIR = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/runs/'

RUNS_INFO_PATH = os.path.join('/lhome/ific/c/ccortesp/Analysis/NEXT-100/Backgrounds/utilities/runs_information.csv')

SUMMARY_FILENAME = 'summary_' + VERSION_TAG +'.csv'     # Choose your name
SUMMARY_PATH = os.path.join('/lhome/ific/c/ccortesp/Analysis/NEXT-100/Backgrounds/txt/', SUMMARY_FILENAME)

# KEYS
DORO_KEY = 'DST/Events'
SOPH_KEY = 'RECO/Events'
EVT_KEY  = 'PROCESSED/Events'

# COLUMNS TO USE
DORO_COLUMNS = ['event', 'time', 'nS1', 'nS2', 'S1h', 'S1e', 'S2e', 'DT', 'X', 'Y', 'Z']
SOPH_COLUMNS = ['event', 'time', 'npeak', 'X', 'Y', 'Z', 'Q', 'E']
FINAL_SOPH_COLUMNS = ['event', 'time', 'npeak', 'X', 'Y', 'DT', 'Z', 'E_hit_pe']

# CUTFLOWS
CUT_NAMES = ['Reconstructed', 'Z_Positive', 'S1_Cut', 'Clean_Hits']

# --------------------------------
# 2. ALPHA/ELECTRON DISCRIMINATION
# --------------------------------
S1_ENERGY_THRESHOLD = 900     # in [PE]
SIZE_THRESHOLD = 2e3          # in [# of hits]

# -------------------
# 3. DETECTOR REGIONS
# -------------------
# Geometric boundaries for event classification.
Z_LOW = 40          # in [mm]
Z_UP  = 1147        # in [mm]
R_UP  = 451.65      # in [mm]

## Runs & Summary Information

In [3]:
# ----- Load Dataframes ----- #
RUNS_INFO_DF = pd.read_csv(RUNS_INFO_PATH)
RUNS_INFO_DF.columns = RUNS_INFO_DF.columns.str.strip()
RUNS_INFO_DF

,run_number,duration,OK,LOST,period,condition
0,15062,84783,69564,1339,1,castle_open
1,15063,79120,65052,1241,1,castle_open
2,15076,69316,56775,1080,1,castle_open
3,15288,87256,30201,8397,1,castle_pclosed_RAS
4,15289,82152,28180,7884,1,castle_pclosed_RAS
...,...,...,...,...,...,...
106,15733,86919,30475,10027,2,castle_closed_RAS
107,15734,85790,29837,9598,2,castle_closed_RAS
108,15735,87451,30547,9958,2,castle_closed_RAS
109,15736,93376,32622,10506,2,castle_closed_RAS


In [4]:
SUMMARY_DF = pd.read_csv(SUMMARY_PATH)
SUMMARY_DF.drop(columns=['Unnamed: 0'], inplace=True)
SUMMARY_DF.columns = SUMMARY_DF.columns.str.strip()
SUMMARY_DF.sort_values(by='Run_ID', inplace=True)
SUMMARY_DF

,Run_ID,Duration,Date_CV,Date_Err,OK,LOST,Reconstructed,Z_Positive,S1_Cut,Clean_Hits
40,15609,79563,1.753150e+09,167.3591,28168,8392,27865,20667,17703,17695
42,15614,81279,1.753244e+09,171.2587,28543,8472,28225,20811,17664,17656
44,15615,86302,1.753328e+09,174.9253,30165,9046,29767,21927,18634,18629
46,15616,86146,1.753414e+09,174.7854,30386,8635,30124,22198,18919,18907
47,15617,88041,1.753502e+09,175.5444,31274,8983,31052,22945,19591,19584
...,...,...,...,...,...,...,...,...,...,...
59,15733,86919,1.758939e+09,175.0674,30475,10027,30144,22120,18663,18657
62,15734,85790,1.759026e+09,176.3823,29837,9598,29513,21813,18275,18261
64,15735,87451,1.759113e+09,175.5591,30547,9958,30193,22198,18810,18801
66,15736,93376,1.759204e+09,181.1218,32622,10506,32243,23718,20280,20272


# Merge by Period & Detector Condition

In [8]:
# ----- Configuration ----- #
DATA_PERIOD = 2                 # Options: 1, 2
DETECTOR_CONDITION = 'castle_closed'       # Options: None, 'castle_open', 'castle_closed', 'castle_closed_RAS', 'castle_pclosed', 'castle_pclosed_RAS'

In [9]:
# Select runs to use according to the notebook configuration
if DATA_PERIOD is not None:
    runs_to_analyze = RUNS_INFO_DF.loc[RUNS_INFO_DF['period'] == DATA_PERIOD, 'run_number'].values
    if DETECTOR_CONDITION is not None:
        runs_to_analyze = RUNS_INFO_DF.loc[(RUNS_INFO_DF['period'] == DATA_PERIOD) & (RUNS_INFO_DF['condition'] == DETECTOR_CONDITION), 'run_number'].values

# Selection
print(f"\nSelected {len(runs_to_analyze)} runs for merge:")
print(runs_to_analyze)


Selected 11 runs for merge:
[15609 15614 15615 15616 15617 15618 15619 15621 15622 15623 15624]


In [10]:
total_corr_time = 0
# total_ok = 0
# total_lost = 0
total_processed_events = 0
all_processed_df = []

for run_id in runs_to_analyze:

    print(f"--- Merging Run {run_id} ---")
    if run_id not in SUMMARY_DF['Run_ID'].values:
        print(f"  --> Run {run_id} not found in summary file. Skipping...")
        continue

    # Run information
    run_duration = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'Duration'].values[0]
    run_OK   = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'OK'].values[0]
    run_LOST = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'LOST'].values[0]
    DAQe_CV, DAQe_error = ff.efficiency(run_OK, run_LOST)
    run_corr_time = run_duration * DAQe_CV
    total_corr_time += run_corr_time

    # Accumulate processed events
    processed_events = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'Clean_Hits'].values[0]
    total_processed_events += processed_events
    # Add run_number for the global_event_id
    run_file = os.path.join(PROCESSED_DIR, f'processed_run_{run_id}_{VERSION_TAG}.h5')
    run_df = pd.read_hdf(run_file, key='Events')
    run_df['run_number'] = run_id
    all_processed_df.append(run_df)

# Compute corrected time
print(f"\nFor period {DATA_PERIOD} with condition '{DETECTOR_CONDITION}':\n  Corrected Time = {total_corr_time} s")

# Concatenate all dataframes
MERGED_DF = pd.concat(all_processed_df, ignore_index=True)
print(f"Dataframe merged:\n  Total processed events: {total_processed_events}")

--- Merging Run 15609 ---
--- Merging Run 15614 ---
--- Merging Run 15615 ---
--- Merging Run 15616 ---
--- Merging Run 15617 ---
--- Merging Run 15618 ---
--- Merging Run 15619 ---
--- Merging Run 15621 ---
--- Merging Run 15622 ---
--- Merging Run 15623 ---
--- Merging Run 15624 ---

For period 2 with condition 'castle_closed':
  Corrected Time = 697798.6254801187 s
Dataframe merged:
  Total processed events: 197775


### Compute Global Event ID

In [11]:
# An original event is defined as a row in dataframe where at least one of the columns 
# ('event', 'run_number') differs from the corresponding row below it (using `shift`).
event_OG = (MERGED_DF[['event', 'run_number']] != MERGED_DF[['event', 'run_number']].shift())

# If any column in event_OG is True, it means the row corresponds to the start of a new original event block.
new_event_block = event_OG.any(axis=1)

# Use `cumsum()` on the boolean mask to create a unique identifier for each contiguous block of hits 
# that belong to the same original event.
unique_block_id = new_event_block.cumsum()

# Assign a unique global event ID to each block of original events.
# The `factorize` function generates a unique integer code for each unique block ID.
MERGED_DF['global_event'] = pd.factorize(unique_block_id)[0]
print(f"{MERGED_DF['global_event'].nunique()} unique global events identified.")

197775 unique global events identified.


In [12]:
MERGED_DF

,event,npeak,E_peak_pe,n_hits_peak,X_bary,Y_bary,Z_bary,Z_min,Z_max,R_max,E_evt_pe,n_hits,time,nS1,nS2,S1e_max,S1e_corr_max,n_hits_original,run_number,global_event
0,729,27,1.088816e+06,7736,64.465804,-198.521221,1184.605694,1164.560204,1205.445619,492.644427,1.088816e+06,7736,1.753110e+09,1,1,1375.999634,1377.983858,8251,15609,0
1,1135,24,1.071127e+06,6951,20.125879,-285.444652,1185.351586,1166.460069,1203.718106,494.657494,1.071127e+06,6951,1.753110e+09,1,1,1288.578125,1289.435936,7591,15609,1
2,2066,22,2.652837e+05,568,56.829979,448.314182,322.837112,315.182645,331.176387,503.637966,2.652837e+05,568,1.753110e+09,1,1,899.579102,1407.285688,1263,15609,2
3,2297,18,2.738256e+05,607,262.824232,361.850465,706.187590,694.058916,720.928844,502.607786,2.738256e+05,607,1.753110e+09,1,1,1148.663696,1437.162886,1175,15609,3
4,3144,30,1.060969e+06,7376,196.628390,-43.770213,1184.714681,1165.832511,1206.631642,492.459741,1.060969e+06,7376,1.753110e+09,1,1,1457.166504,1458.044011,7937,15609,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209908,1554427,31,1.538204e+05,349,302.245349,340.440730,400.622225,391.827376,411.444927,500.128125,1.538204e+05,349,1.754037e+09,1,1,900.591003,1340.967719,699,15624,197770
209909,1554455,27,1.347803e+05,339,232.762543,-396.895344,779.189804,766.331396,796.838108,499.618561,1.347803e+05,339,1.754037e+09,1,1,1088.411743,1312.204418,595,15624,197771
209910,1554567,25,2.929788e+05,583,460.451347,-17.398265,228.905920,220.902835,240.686358,498.898292,2.929788e+05,583,1.754037e+09,1,1,797.864563,1329.433384,1311,15624,197772
209911,1554910,21,1.033557e+05,196,-380.598360,-262.627481,228.273411,222.587963,235.062777,491.851900,1.033557e+05,196,1.754037e+09,1,1,642.648376,1071.320454,594,15624,197773


# Tagging Events

### By Particle

In [13]:
particle_tagged_MERGED_DF = bf.tag_particles(MERGED_DF, size_threshold=SIZE_THRESHOLD, s1_energy_threshold=S1_ENERGY_THRESHOLD, event_column='global_event')

In [14]:
particle_tagged_MERGED_DF

,event,npeak,E_peak_pe,n_hits_peak,X_bary,Y_bary,Z_bary,Z_min,Z_max,R_max,...,n_hits,time,nS1,nS2,S1e_max,S1e_corr_max,n_hits_original,run_number,global_event,particle
0,729,27,1.088816e+06,7736,64.465804,-198.521221,1184.605694,1164.560204,1205.445619,492.644427,...,7736,1.753110e+09,1,1,1375.999634,1377.983858,8251,15609,0,alpha
1,1135,24,1.071127e+06,6951,20.125879,-285.444652,1185.351586,1166.460069,1203.718106,494.657494,...,6951,1.753110e+09,1,1,1288.578125,1289.435936,7591,15609,1,alpha
2,2066,22,2.652837e+05,568,56.829979,448.314182,322.837112,315.182645,331.176387,503.637966,...,568,1.753110e+09,1,1,899.579102,1407.285688,1263,15609,2,alpha
3,2297,18,2.738256e+05,607,262.824232,361.850465,706.187590,694.058916,720.928844,502.607786,...,607,1.753110e+09,1,1,1148.663696,1437.162886,1175,15609,3,alpha
4,3144,30,1.060969e+06,7376,196.628390,-43.770213,1184.714681,1165.832511,1206.631642,492.459741,...,7376,1.753110e+09,1,1,1457.166504,1458.044011,7937,15609,4,alpha
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209908,1554427,31,1.538204e+05,349,302.245349,340.440730,400.622225,391.827376,411.444927,500.128125,...,349,1.754037e+09,1,1,900.591003,1340.967719,699,15624,197770,alpha
209909,1554455,27,1.347803e+05,339,232.762543,-396.895344,779.189804,766.331396,796.838108,499.618561,...,339,1.754037e+09,1,1,1088.411743,1312.204418,595,15624,197771,alpha
209910,1554567,25,2.929788e+05,583,460.451347,-17.398265,228.905920,220.902835,240.686358,498.898292,...,583,1.754037e+09,1,1,797.864563,1329.433384,1311,15624,197772,alpha
209911,1554910,21,1.033557e+05,196,-380.598360,-262.627481,228.273411,222.587963,235.062777,491.851900,...,196,1.754037e+09,1,1,642.648376,1071.320454,594,15624,197773,alpha


### By Detector Region

In [15]:
region_tagged_MERGED_DF = bf.tag_event_by_detector_region(particle_tagged_MERGED_DF, z_cut_low=Z_LOW, z_cut_high=Z_UP, r_cut_high=R_UP, event_column='global_event')

In [16]:
region_tagged_MERGED_DF

,event,npeak,E_peak_pe,n_hits_peak,X_bary,Y_bary,Z_bary,Z_min,Z_max,R_max,...,time,nS1,nS2,S1e_max,S1e_corr_max,n_hits_original,run_number,global_event,particle,region
0,729,27,1.088816e+06,7736,64.465804,-198.521221,1184.605694,1164.560204,1205.445619,492.644427,...,1.753110e+09,1,1,1375.999634,1377.983858,8251,15609,0,alpha,cathode
1,1135,24,1.071127e+06,6951,20.125879,-285.444652,1185.351586,1166.460069,1203.718106,494.657494,...,1.753110e+09,1,1,1288.578125,1289.435936,7591,15609,1,alpha,cathode
2,2066,22,2.652837e+05,568,56.829979,448.314182,322.837112,315.182645,331.176387,503.637966,...,1.753110e+09,1,1,899.579102,1407.285688,1263,15609,2,alpha,tube
3,2297,18,2.738256e+05,607,262.824232,361.850465,706.187590,694.058916,720.928844,502.607786,...,1.753110e+09,1,1,1148.663696,1437.162886,1175,15609,3,alpha,tube
4,3144,30,1.060969e+06,7376,196.628390,-43.770213,1184.714681,1165.832511,1206.631642,492.459741,...,1.753110e+09,1,1,1457.166504,1458.044011,7937,15609,4,alpha,cathode
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209908,1554427,31,1.538204e+05,349,302.245349,340.440730,400.622225,391.827376,411.444927,500.128125,...,1.754037e+09,1,1,900.591003,1340.967719,699,15624,197770,alpha,tube
209909,1554455,27,1.347803e+05,339,232.762543,-396.895344,779.189804,766.331396,796.838108,499.618561,...,1.754037e+09,1,1,1088.411743,1312.204418,595,15624,197771,alpha,tube
209910,1554567,25,2.929788e+05,583,460.451347,-17.398265,228.905920,220.902835,240.686358,498.898292,...,1.754037e+09,1,1,797.864563,1329.433384,1311,15624,197772,alpha,tube
209911,1554910,21,1.033557e+05,196,-380.598360,-262.627481,228.273411,222.587963,235.062777,491.851900,...,1.754037e+09,1,1,642.648376,1071.320454,594,15624,197773,alpha,tube


# Output

In [17]:
# H5 output filename
MERGED_FILENAME = f'merged_tagged_runs_{DETECTOR_CONDITION}_{VERSION_TAG}.h5'
MERGED_PATH = os.path.join('/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/', MERGED_FILENAME)
print(f"\nSaving merged dataframe to: {MERGED_PATH}")


Saving merged dataframe to: /lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/merged_tagged_runs_castle_closed_p2_final.h5


In [18]:
region_tagged_MERGED_DF.to_hdf(MERGED_PATH, key='Events', mode='w', format='table')
print('Done!')

Done!
